# 7장 4차시 실습

`sales_clean.csv`로 총액 회귀 모델을 비교하고, CO2 시계열 예측과 Wine PCA를 실행합니다.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import platform
from sklearn.model_selection import train_test_split, cross_val_score

fonts = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}
plt.rcParams["font.family"] = fonts.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

## ① 읽고 나눈다 : 총액을 수량과 단가로

In [2]:
sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## ② linear regression 과 decision tree

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

lin = LinearRegression().fit(X_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression  test R² :", round(lin.score(X_test, y_test), 3))
print("decision tree      test R² :", round(tree.score(X_test, y_test), 3))

linear regression  test R² : 0.862
decision tree      test R² : 0.994


## ③ 특징 하나를 더한다 : 수량 × 단가

In [4]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin2 = LinearRegression().fit(X2_train, y_train)
print("linear + qp        test R² :", round(lin2.score(X2_test, y_test), 3))
print("coef :", lin2.coef_.round(3), " intercept :", round(lin2.intercept_, 1))

linear + qp        test R² : 1.0
coef : [0. 0. 1.]  intercept : -0.0


## ④ max_depth 를 바꾸면 : training 과 test

In [5]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## ⑤ 교차검증 : 나눔 한 번을 믿지 않는다

In [6]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.6f}  std {s.std():.6f}")

max_depth 3     mean 0.845258  std 0.019086
max_depth 5     mean 0.960743  std 0.002673
max_depth 8     mean 0.991885  std 0.000397
max_depth 12    mean 0.994620  std 0.000522


max_depth None  mean 0.994721  std 0.000618


**깊이 선택**

깊이 8의 교차검증 평균은 0.991885이며 깊이 12의 0.994620과 약 0.0027 차이입니다.
잎 수가 250개인 깊이 8을 선택합니다. 깊이 12의 잎 781개보다 적은 분기로 비슷한 점수를 얻습니다.